# DDoS attack-HOIC - encoder generator

## Import libraries

In [6]:
import torch
from torch import nn, optim
import torch.nn.functional as F
import numpy as np
import pandas as pd

## Import training, validation and test data

In [11]:
train = pd.read_csv('../../train.csv')
val = pd.read_csv('../../val.csv')
test = pd.read_csv('../../test.csv')

'Infilteration' and 'DoS attacks-SlowHTTPTest' classes were aparently mislabeled, so we will exclude them, remaining 12 attack classes

In [12]:
classes_out = ['Infilteration','DoS attacks-SlowHTTPTest']
train = train.drop(index=train[train['Label'].isin(classes_out)].index)
val = val.drop(index=val[val['Label'].isin(classes_out)].index)
test = test.drop(index=test[test['Label'].isin(classes_out)].index)

11 attack classes will be joined into major attack classes
* FTP-BruteForce and SSH-Bruteforce -> BruteForce
* DoS attacks-GoldenEye, DoS attacks-Slowloris and DoS attacks-Hulk -> DoS
* DDoS attacks-LOIC-HTTP, DDOS attack-LOIC-UDP and DDOS attack-HOIC -> DDoS
* Brute Force -Web, Brute Force -XSS and SQL Injection -> DDoS

Resulting in 5 major attack classes: BruteForce, DoS, DDoS, Web and Bot, and Benign class 

In [13]:
train['Label'] = train['Label'].replace('FTP-BruteForce','BruteForce')
train['Label'] = train['Label'].replace('SSH-Bruteforce','BruteForce')
train['Label'] = train['Label'].replace('DoS attacks-GoldenEye','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Slowloris','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Hulk','DoS')
train['Label'] = train['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
# train['Label'] = train['Label'].replace('DDOS attack-HOIC','DDoS')
train['Label'] = train['Label'].replace('Brute Force -Web','Web')
train['Label'] = train['Label'].replace('Brute Force -XSS','Web')
train['Label'] = train['Label'].replace('SQL Injection','Web')


val['Label'] = val['Label'].replace('FTP-BruteForce','BruteForce')
val['Label'] = val['Label'].replace('SSH-Bruteforce','BruteForce')
val['Label'] = val['Label'].replace('DoS attacks-GoldenEye','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Slowloris','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Hulk','DoS')
val['Label'] = val['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
# val['Label'] = val['Label'].replace('DDOS attack-HOIC','DDoS')
val['Label'] = val['Label'].replace('Brute Force -Web','Web')
val['Label'] = val['Label'].replace('Brute Force -XSS','Web')
val['Label'] = val['Label'].replace('SQL Injection','Web')


test['Label'] = test['Label'].replace('FTP-BruteForce','BruteForce')
test['Label'] = test['Label'].replace('SSH-Bruteforce','BruteForce')
test['Label'] = test['Label'].replace('DoS attacks-GoldenEye','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Slowloris','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Hulk','DoS')
test['Label'] = test['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
# test['Label'] = test['Label'].replace('DDOS attack-HOIC','DDoS')
test['Label'] = test['Label'].replace('Brute Force -Web','Web')
test['Label'] = test['Label'].replace('Brute Force -XSS','Web')
test['Label'] = test['Label'].replace('SQL Injection','Web')

## Labels into numeric

In [14]:
labels_str = [i for i in train['Label'].value_counts().index]
print(labels_str)

for i, l in enumerate(labels_str):
    train['Label'] = train['Label'].replace(l, i)
    val['Label'] = val['Label'].replace(l, i)
    test['Label'] = test['Label'].replace(l, i)

['Benign', 'DDOS attack-HOIC', 'DDoS', 'DoS', 'BruteForce', 'Bot', 'Web']


C:\Users\Lincoln\AppData\Local\Temp\ipykernel_332\2307520911.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['Label'] = train['Label'].replace(l, i)
C:\Users\Lincoln\AppData\Local\Temp\ipykernel_332\2307520911.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val['Label'] = val['Label'].replace(l, i)
C:\Users\Lincoln\AppData\Local\Temp\ipykernel_332\2307520911.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result

## Contrastive AutoEncoder Functions

<img src="..\..\img\contrastive_pairs.png" alt="Contrastive Pairs" width="60%"/>

In [ ]:
def pick_pair(embeddings, labels, pick_embeddings, pick_labels):
    """
    Generates all pairs between embeddings (batch) and pick_embeddings (reference samples), with labels_pair indicating whether the pairs are positive (0) or negative (1). 
    """
    N, D = embeddings.shape
    M = pick_embeddings.shape[0]

    # Expands to all possible combinations
    A = embeddings.unsqueeze(0).expand(M, N, D).reshape(M * N, D)
    B = pick_embeddings.unsqueeze(1).expand(M, N, D).reshape(M * N, D)

    labels_exp = labels.unsqueeze(0).expand(M, N)
    pick_labels_exp = pick_labels.unsqueeze(1).expand(M, N)
    labels_pair = (labels_exp != pick_labels_exp).reshape(-1).float()

    return A, B, labels_pair

def ContrastiveLoss(input1, input2, y, margin=1.0):
    # y = 0 when input1 and input2 belong to the same class, and y = 1 otherwise.
    diff = input1 - input2
    dist_sq = torch.sum(torch.pow(diff,2),1)
    dist = torch.sqrt(dist_sq + 1e-10) # Adds 1e-10 because if the sqrt is 0, the gradient becomes NaN.
    mdist = margin - dist
    mdist = F.relu(mdist)
    loss = ((1-y) * dist * dist) + (y * mdist * mdist)
    return torch.mean(loss)

def AEContrastive(input, output, embeddings, labels, pick_encoded, pick_labels, margin=1.0, lambda1=1.0):
    mseloss = nn.MSELoss()
    ae_loss = mseloss(input,output)

    A, B, labels_pair = pick_pair(embeddings, labels, pick_encoded, pick_labels)
    contrastive_loss = ContrastiveLoss(A, B, labels_pair, margin=margin)
    return ae_loss + contrastive_loss * lambda1, ae_loss, contrastive_loss

In [ ]:
class contrastive_ae(nn.Module):
    def __init__(self, input_neurons, neurons):
        super().__init__()
        self.encoder0 = nn.Linear(input_neurons,64)
        self.encoder1 = nn.Linear(64,32)
        self.encoder2 = nn.Linear(32,16)
        self.encoder3 = nn.Linear(16,neurons)

        self.decoder0 = nn.Linear(neurons,16)
        self.decoder1 = nn.Linear(16,32)
        self.decoder2 = nn.Linear(32,64)
        self.decoder3 = nn.Linear(64,input_neurons)

        self.activation0 = nn.ReLU()
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X) # No activation function
        encoded = X
        X = self.activation0(self.decoder0(X))
        X = self.activation0(self.decoder1(X))
        X = self.activation0(self.decoder2(X))
        X = self.decoder3(X) # No activation function


        return X, encoded

In [6]:
class encoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder0 = list(model.children())[0]
        self.encoder1 = list(model.children())[1]
        self.encoder2 = list(model.children())[2]
        self.encoder3 = list(model.children())[3]
        self.activation0 = list(model.children())[8]
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        return X

In [ ]:
from torch.utils.data import DataLoader,TensorDataset


# Prepare reference samples for each batch

def build_class_reference_batches(X_train, y_train, batch_size=512):
    classes = torch.unique(y_train).tolist()
    n_classes = len(classes)
    total_samples = len(X_train)
    dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    

    print(classes)

    # Maps each class to its indices
    class_to_indices = {
        int(cls.item()): (y_train == cls).nonzero(as_tuple=True)[0].tolist()
        for cls in torch.unique(y_train)
    }

    reference_list = []
    total_batches = len(train_loader)

    for batch_idx in range(total_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, total_samples)
        batch_indices = set(range(start, end))

        class_samples = []
        for cls in classes:
            candidates = [i for i in class_to_indices[cls] if i not in batch_indices]
            if not candidates:
                raise ValueError(f"Class {cls} has no samples outside of batch {batch_idx}")
            chosen = candidates[torch.randint(len(candidates), (1,)).item()]
            class_samples.append(chosen)

        reference_list.append(class_samples)


    return train_loader, reference_list

In [ ]:
# Normalization function


def normalize(t):
    t_norm = (t - t.mean()) / t.std()
    return t_norm


In [ ]:
array_hidden_classes = [[1]] # Class 1 correspond to DDoS attack-HOIC class
filenames = ['1']

total_classes = len(labels_str)
for i in range(len(array_hidden_classes)):
    filename = f'{filenames[i]}_hidden'
    print(filename)
    hidden_classes = array_hidden_classes[i] # Classes hidden from training

    hidden_classes_index = list(train[train['Label'].isin(hidden_classes)].index)

    train_hidden = train.loc[hidden_classes_index]

    train_not_hidden = train.drop(hidden_classes_index)

    X_train = train_not_hidden.drop(columns=['Label'])
    y_train = train_not_hidden['Label']
    X_val = val.drop(columns=['Label'])
    y_val = val['Label'].values
    X_test = test.drop(columns=['Label'])
    y_test = test['Label'].values

    torch.manual_seed(123)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)
    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_loader, pick_list = build_class_reference_batches(X_train,y_train, batch_size=512)
    pick_samples = []
    pick_labels = []
    for l in pick_list:
        pick_samples.append(X_train[l])
        pick_labels.append(y_train[l])

    pick_samples = torch.stack(pick_samples)    
    pick_labels = torch.stack(pick_labels)


    neurons = total_classes - len(hidden_classes)
    print(f'{X_train.shape[1]} {neurons}')
    model = contrastive_ae(input_neurons=X_train.shape[1],neurons=neurons)
    model.to(device)

    learning_rate = 0.0001
    opt = optim.Adam(model.parameters(),lr=learning_rate)
    margin = 10 # Class separation parameter
    cae_lambda_1 = 1.0 # Contrastive learning weight 
    
    
    for epoch in range(250):
        running_loss = 0
        running_ael = 0
        running_cl = 0
        counter = 0
        for data in train_loader:
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)
            ps = pick_samples[counter]
            pl = pick_labels[counter]
            ps = ps.to(device)
            pl = pl.to(device)
            counter += 1
            opt.zero_grad()
            outputs, encoded = model(inputs)
            _, pick_encoded = model(ps)
            tam_pe = pick_encoded.shape[0]
            enc_total = torch.cat((encoded,pick_encoded), dim=0)
            enc_total_norm = normalize(enc_total)
            encoded_norm = enc_total_norm[:-tam_pe]
            pick_encoded_norm = enc_total_norm[-tam_pe:]
            loss, ael, cl = AEContrastive(input=inputs, output=outputs, embeddings = encoded_norm, labels=labels, pick_encoded = pick_encoded_norm, pick_labels = pl, margin=margin, lambda1=cae_lambda_1)
            loss.backward()
            opt.step()
            running_loss += loss.item()
            running_ael += ael.item()
            running_cl += cl.item()
        print(f'filename: {filename} Epoch {epoch+1} loss:{running_loss/len(train_loader)} ael:{running_ael/len(train_loader)} cl:{running_cl/len(train_loader)}')

    model_e = encoder(model=model)
    model_e.to(device)

    model_e.eval()

    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_encoded = model_e(X_train.to(device))

    train_encoded = train_encoded.detach().cpu().numpy()

    train_encoded_df = pd.DataFrame(train_encoded)
    train_encoded_df['Label'] = y_train.detach().cpu().numpy().astype(int)
    display(train_encoded_df)


    X_val = torch.tensor(np.array(X_val), dtype=torch.float)

    val_encoded = model_e(X_val.to(device))

    val_encoded = val_encoded.detach().cpu().numpy()

    val_encoded_df = pd.DataFrame(val_encoded)

    val_encoded_df['Label'] = y_val

    display(val_encoded_df)

    X_test = torch.tensor(np.array(X_test), dtype=torch.float)

    test_encoded = model_e(X_test.to(device))

    test_encoded = test_encoded.detach().cpu().numpy()

    test_encoded_df = pd.DataFrame(test_encoded)

    test_encoded_df['Label'] = y_test

    display(test_encoded_df)

    train_encoded_df.to_csv(f'train_encoded_{filename}.csv',index=False)
    val_encoded_df.to_csv(f'val_encoded_{filename}.csv',index=False)
    test_encoded_df.to_csv(f'test_encoded_{filename}.csv',index=False)

1_hidden


cuda


[0.0, 2.0, 3.0, 4.0, 5.0, 6.0]


pick completo
82 6


filename: 1_hidden Epoch 1 loss:3.5873888595241836 ael:0.02484512998549596 cl:35.62543678105156


...

/tmp/ipykernel_739336/512526955.py:91: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  X_train = torch.tensor(np.array(X_train), dtype=torch.float)
/tmp/ipykernel_739336/512526955.py:92: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  y_train = torch.tensor(np.array(y_train), dtype=torch.float)


,0,1,2,3,4,5,Label
0,0.058388,0.061334,0.052688,0.059286,0.018206,0.056789,0
1,0.053023,0.021907,0.048774,0.056414,0.065297,0.022233,3
2,0.048704,0.058515,0.040151,0.043024,0.014153,0.056211,0
3,0.085507,0.081204,0.097371,0.103960,0.075657,0.045728,5
4,0.053023,0.021906,0.048773,0.056414,0.065294,0.022230,3
...,...,...,...,...,...,...,...
1640803,0.045622,0.045222,0.064377,0.048756,0.071089,0.091958,2
1640804,0.053023,0.021908,0.048774,0.056414,0.065298,0.022234,3
1640805,0.053023,0.021907,0.048774,0.056414,0.065297,0.022233,3
1640806,0.053023,0.021907,0.048774,0.056414,0.065298,0.022234,3


Davies-Bouldin Score: 1.8121724486195447


,0,1,2,3,4,5,Label
0,0.045873,0.045481,0.064604,0.049166,0.070915,0.091649,2
1,0.053027,0.021909,0.048786,0.056424,0.065306,0.022238,3
2,0.056866,0.061526,0.056227,0.059583,0.027283,0.057446,0
3,0.063447,0.098116,0.023033,0.032556,0.077840,0.045450,4
4,0.055863,0.061792,0.057420,0.058792,0.022750,0.056634,0
...,...,...,...,...,...,...,...
519951,0.055251,0.061649,0.050665,0.055196,0.017390,0.056058,0
519952,0.085177,0.081428,0.101228,0.106954,0.077557,0.052935,5
519953,0.042423,0.042624,0.055057,0.048819,0.021929,0.078310,1
519954,0.057042,0.059628,0.053075,0.058165,0.023129,0.057145,0


,0,1,2,3,4,5,Label
0,0.042392,0.042797,0.055168,0.048688,0.023354,0.078836,1
1,0.045308,0.044898,0.064096,0.048245,0.071309,0.092344,2
2,0.056807,0.060230,0.056812,0.060584,0.024277,0.057017,0
3,0.051356,0.054295,0.048205,0.050696,0.018832,0.059108,0
4,0.056940,0.050340,0.070786,0.066723,0.058604,0.060848,1
...,...,...,...,...,...,...,...
649942,0.056217,0.061969,0.056903,0.058777,0.023097,0.055188,0
649943,0.058325,0.051064,0.073039,0.068674,0.060766,0.060577,1
649944,0.057610,0.050110,0.072318,0.067674,0.060891,0.060806,1
649945,0.063620,0.096664,0.024867,0.034103,0.076641,0.045543,4
